Exploratory Visualization: Fishery Landings and Climate Lag Effect

Hypothesis (Lag Effect/Species Biology): For some species, commercial fishery landings in California are more strongly associated with climate conditions observed several years earlier than with climate conditions in the same year.

This notebook explores this hypothesis for one focal species: California halibut. Because halibut require 4 to 5 years to grow large enough to be harvested by the commercial fishery, favorable ocean conditions for egg and larval survival may influence landings only after a biologically meaningful time lag.

This notebook explores that hypothesis by comparing annual commercial landings with annual mean RONI under both simultaneous and lagged climate scenarios.

Background: Positive RONI values correspond to relatively warm ocean conditions, while negative values correspond to relatively cool conditions. California halibut egg and larval is generally considered to be positively associated with warm-water years, suggesting that favorable climate conditions may influence commercial landings only after juveniles recruit into the fishery.

In [118]:
import pandas as pd
import numpy as np
import altair as alt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 20)

alt.data_transformers.disable_max_rows()

halibut = pd.read_csv(
    "../data/processed/halibut.csv"
)

roni_annual = pd.read_csv(
    "../data/processed/roni_annual.csv"
)

halibut_climate = pd.read_csv("../data/processed/halibut_climate.csv")

mfde = pd.read_csv("../data/processed/mfde_clean.csv")

In [119]:
halibut_climate.head()

,Year,Pounds,Value,Annual_RONI
0,1980,726550.0,1000982.0,0.241667
1,1981,1262018.0,1874168.0,-0.208333
2,1982,1212312.0,1745829.0,1.175000
3,1983,1130363.0,1611535.0,0.475000
4,1984,1107019.0,1855602.0,-0.441667


In [120]:
alt.Chart(halibut_climate).mark_circle(size=80).encode(
    x=alt.X("Annual_RONI:Q", title="Annual Mean RONI"),
    y=alt.Y("Pounds:Q", title="California Halibut Landings (lbs)"),
    tooltip=["Year", "Annual_RONI", "Pounds"]
).properties(
    title="No-Lag Comparison: Annual RONI vs. California Halibut Landings",
    width=600,
    height=400
)

alt.Chart(...)

Is there an observable relationship between annual climate conditions (RONI) and annual California halibut landings when no lag is applied?

The scatterplot shows no obvious linear relationship between annual mean RONI and California halibut landings. Years with both high and low landings occur across nearly the full range of RONI values, suggesting that climate conditions alone do not explain annual fishery production.

California halibut require several years to grow before recruiting to the commercial fishery. Therefore, climate conditions during the landing year may not reflect the environmental conditions experienced by the fish entering the fishery. This observation motivates exploration of lagged climate relationships.

In [121]:
points = alt.Chart(halibut_climate).mark_circle(size=80).encode(
    x=alt.X("Annual_RONI:Q", title="Annual Mean RONI"),
    y=alt.Y("Pounds:Q", title="California Halibut Landings (lbs)"),
    tooltip=["Year", "Annual_RONI", "Pounds"]
)

trend = (
    alt.Chart(halibut_climate)
    .transform_regression("Annual_RONI", "Pounds")
    .mark_line(color="red")
    .encode(
        x="Annual_RONI:Q",
        y="Pounds:Q"
    )
)

points + trend

alt.LayerChart(...)

### Observation

Adding a regression line to the no-lag scatterplot suggests only a weak positive relationship between annual mean RONI and California halibut landings. The points remain widely scattered, indicating that same-year climate conditions are unlikely to fully explain variation in landings. This motivated the next step: testing whether climate conditions from prior years show a stronger relationship with landings.

In [172]:
correlations = []

for lag in range(0, 11):

    # Shift RONI forward in time
    roni_lag = roni_annual.copy()
    roni_lag["Year"] = roni_lag["Year"] + lag
    roni_lag = roni_lag.rename(columns={"Annual_RONI": "RONI_lagged"})

    # Merge with California halibut landings
    temp = halibut.merge(
        roni_lag,
        on="Year",
        how="inner"
    )

    corr = temp["Pounds"].corr(temp["RONI_lagged"])

    correlations.append({
        "Lag": lag,
        "Correlation": corr,
        "Years": len(temp)
    })

corr_df = pd.DataFrame(correlations)

corr_df

,Lag,Correlation,Years
0,0,0.107383,46
1,1,0.069386,46
2,2,0.148665,46
3,3,0.038282,46
4,4,0.192426,46
5,5,0.299710,46
6,6,0.281690,46
7,7,0.270220,46
8,8,0.129822,46
9,9,0.133494,46


In [173]:
alt.Chart(corr_df).mark_line(point=True).encode(
    x=alt.X("Lag:Q", title="Lag (Years)"),
    y=alt.Y("Correlation:Q", title="Pearson Correlation")
).properties(
    title="Correlation Between Annual RONI and California Halibut Landings by Lag",
    width=600,
    height=350
)

alt.Chart(...)

Exploratory Observation

The initial lag analysis demonstrated that accounting for delayed biological responses substantially strengthened the observed relationship between climate and California halibut landings. After extending the climate record to include pre-1980 RONI values and applying lag by shifting climate years prior to merging with the fisheries data, the strongest Pearson correlation occurred at approximately five years (r ≈ 0.30). Correlations remained elevated through approximately seven years before gradually declining, suggesting that California halibut respond to favorable ocean conditions over a broader 4–7 year response window rather than at a single discrete lag.

In [174]:
best_lag = 5

roni_lag = roni_annual.copy()
roni_lag["Year"] = roni_lag["Year"] + best_lag
roni_lag = roni_lag.rename(columns={"Annual_RONI": "RONI_lag5"})

halibut_lag5 = halibut.merge(
    roni_lag,
    on="Year",
    how="inner"
)

halibut_lag5.head()

,Year,Pounds,Value,RONI_lag5
0,1980,726550.0,1000982.0,-0.700000
1,1981,1262018.0,1874168.0,0.300000
2,1982,1212312.0,1745829.0,0.691667
3,1983,1130363.0,1611535.0,0.025000
4,1984,1107019.0,1855602.0,0.258333


In [175]:
points = alt.Chart(halibut_lag5).mark_circle(size=80).encode(
    x=alt.X(
        "RONI_lag5:Q",
        title="Annual Mean RONI (5-Year Lag)"
    ),
    y=alt.Y(
        "Pounds:Q",
        title="California Halibut Landings (lbs)"
    ),
    tooltip=[
        "Year",
        alt.Tooltip("RONI_lag5:Q", format=".2f"),
        alt.Tooltip("Pounds:Q", format=",.0f")
    ]
)

trend = (
    alt.Chart(halibut_lag5)
    .transform_regression(
        "RONI_lag5",
        "Pounds"
    )
    .mark_line(color="red")
    .encode(
        x="RONI_lag5:Q",
        y="Pounds:Q"
    )
)

(points + trend).properties(
    title="Five-Year Lag Relationship Between RONI and California Halibut Landings",
    width=650,
    height=400
)

alt.LayerChart(...)

To further investigate the lag analysis, annual California halibut landings were compared with annual mean RONI values shifted forward by five years, corresponding to the strongest correlation identified in the previous analysis. Relative to the no-lag comparison, the regression line exhibited a steeper positive slope, indicating a stronger positive association between historical ocean climate conditions and subsequent commercial landings.

Although considerable variability remains, the five-year lag relationship is noticeably stronger than the simultaneous relationship, supporting the hypothesis that California halibut landings respond to ocean climate conditions after a biologically meaningful delay rather than within the same year.

These exploratory findings suggest that incorporating lagged climate variables may improve the investigation of climate-fishery relationships and motivate the development of an interactive visualization that allows users to explore different lag periods across species.

In [176]:
time_series = halibut_climate.copy()

time_series["Landings_z"] = (
    time_series["Pounds"] -
    time_series["Pounds"].mean()
) / time_series["Pounds"].std()

time_series["RONI_z"] = (
    time_series["Annual_RONI"] -
    time_series["Annual_RONI"].mean()
) / time_series["Annual_RONI"].std()

time_series.head()

,Year,Pounds,Value,Annual_RONI,Landings_z,RONI_z
0,1980,726550.0,1000982.0,0.241667,-0.422031,0.402524
1,1981,1262018.0,1874168.0,-0.208333,1.394873,-0.264946
2,1982,1212312.0,1745829.0,1.175000,1.226215,1.786905
3,1983,1130363.0,1611535.0,0.475000,0.948152,0.748619
4,1984,1107019.0,1855602.0,-0.441667,0.868943,-0.611041


Time Series Exploration

While scatterplots and correlation coefficients summarize the overall relationship between climate conditions and commercial fishery landings, they do not illustrate how these variables change through time. To better understand the temporal dynamics of the relationship, annual California halibut landings and annual mean RONI were standardized using z-scores and plotted as time series. Standardization places both variables on a common scale, allowing their relative peaks and troughs to be compared despite their different units of measurement.

The first time series compares annual landings with simultaneous (same-year) RONI values. The second repeats the comparison after applying the five-year lag identified during the correlation analysis. Together, these visualizations provide a qualitative assessment of whether lagging the climate signal improves alignment between periods of favorable ocean conditions and subsequent fishery performance.

In [177]:
plot_df = time_series.melt(
    id_vars="Year",
    value_vars=["Landings_z", "RONI_z"],
    var_name="Variable",
    value_name="Z_score"
)

In [178]:
alt.Chart(plot_df).mark_line(point=True).encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y(
        "Z_score:Q",
        title="Standardized Value (z-score)"
    ),
    color=alt.Color(
        "Variable:N",
        title=""
    ),
    tooltip=["Year", "Variable", "Z_score"]
).properties(
    title="Annual California Halibut Landings and Annual Mean RONI",
    width=700,
    height=400
)

alt.Chart(...)

In [179]:
halibut_lag5.columns

Index(['Year', 'Pounds', 'Value', 'RONI_lag5'], dtype='str')

In [180]:
halibut_lag5["Landings_z"] = (
    halibut_lag5["Pounds"] -
    halibut_lag5["Pounds"].mean()
) / halibut_lag5["Pounds"].std()

halibut_lag5["RONI_z"] = (
    halibut_lag5["RONI_lag5"] -
    halibut_lag5["RONI_lag5"].mean()
) / halibut_lag5["RONI_lag5"].std()

In [181]:
halibut_lag5.columns

Index(['Year', 'Pounds', 'Value', 'RONI_lag5', 'Landings_z', 'RONI_z'], dtype='str')

In [182]:
plot_df_lag5 = halibut_lag5.melt(
    id_vars="Year",
    value_vars=["Landings_z", "RONI_z"],
    var_name="Variable",
    value_name="Z_score"
)

In [183]:
alt.Chart(plot_df_lag5).mark_line(point=True).encode(
    x=alt.X("Year:O", title="Year"),
    y=alt.Y(
        "Z_score:Q",
        title="Standardized Value (z-score)"
    ),
    color=alt.Color(
        "Variable:N",
        title=""
    ),
    tooltip=[
        "Year",
        "Variable",
        alt.Tooltip("Z_score:Q", format=".2f")
    ]
).properties(
    title="California Halibut Landings and Five-Year Lagged Annual Mean RONI",
    width=700,
    height=400
)

alt.Chart(...)

Observations

The no-lag time series showed little visual correspondence between annual mean RONI and California halibut landings, consistent with the weak correlation observed in the initial scatterplot and regression analysis. After applying a five-year lag to the climate index, periods of elevated and depressed RONI appeared to align more closely with subsequent increases and decreases in commercial landings. Although the correspondence is not exact, the lagged comparison provides additional visual evidence that California halibut landings are more closely associated with historical climate conditions than with climate conditions occurring in the same year.

The time series complements the previous statistical analyses by illustrating the temporal sequence underlying the lag hypothesis. Rather than suggesting an immediate response to changing ocean conditions, the visualization supports the interpretation that favorable climate conditions may influence recruitment, growth, and eventual availability to the commercial fishery over several years. This reinforces the biological plausibility of incorporating lagged climate variables when investigating climate–fishery relationships and provides a foundation for an interactive visualization in which users can explore species-specific lag effects.

In [184]:
base = alt.Chart(halibut_lag5).encode(
    x=alt.X(
        "Year:O",
        title="Landing Year",
        axis=alt.Axis(labelAngle=-45)
    )
)

bars = base.mark_bar(opacity=0.75).encode(
    y=alt.Y(
        "Pounds:Q",
        title="California Halibut Landings (lbs)",
        axis=alt.Axis(titleColor="#1f77b4", labelColor="#1f77b4", format="~s")
    ),
    color=alt.value("#4C78A8"),
    tooltip=[
        "Year",
        alt.Tooltip("Pounds:Q", format=",.0f")
    ]
)

roni_line = base.mark_line(point=True, strokeWidth=2).encode(
    y=alt.Y(
        "RONI_lag5:Q",
        title="Annual Mean RONI (5-Year Lag)",
        axis=alt.Axis(titleColor="#d62728", labelColor="#d62728")
    ),
    color=alt.value("#D62728"),
    tooltip=[
        "Year",
        alt.Tooltip("RONI_lag5:Q", format=".2f")
    ]
)

alt.layer(
    bars,
    roni_line
).resolve_scale(
    y="independent"
).properties(
    title="California Halibut Landings and Five-Year Lagged RONI",
    width=800,
    height=400
)

alt.LayerChart(...)

In [185]:
# Build one dataset containing all lag options
lagged_list = []

for lag in range(0, 11):
    roni_lag = roni_annual.copy()
    roni_lag["Year"] = roni_lag["Year"] + lag
    roni_lag = roni_lag.rename(columns={"Annual_RONI": "RONI_lagged"})

    temp = halibut.merge(roni_lag, on="Year", how="inner")
    temp["Lag"] = lag
    lagged_list.append(temp)

halibut_all_lags = pd.concat(lagged_list, ignore_index=True)

In [186]:
lag_slider = alt.param(
    name="lag_years",
    value=7,
    bind=alt.binding_range(
        min=0,
        max=10,
        step=1,
        name="Climate lag in years: "
    )
)

lag_text = (
    alt.Chart(halibut_all_lags)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .transform_aggregate(groupby=["Lag"])
    .transform_calculate(
        label="'Selected lag: ' + datum.Lag + ' years'"
    )
    .mark_text(align="left", fontSize=16, fontWeight="bold")
    .encode(text="label:N")
    .properties(width=800, height=30)
)

base = (
    alt.Chart(halibut_all_lags)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .encode(
        x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45))
    )
)

bars = base.mark_bar(opacity=0.75).encode(
    y=alt.Y(
        "Pounds:Q",
        title="California Halibut Landings (lbs)",
        axis=alt.Axis(format="~s")
    ),
    color=alt.value("#4C78A8"),
    tooltip=[
        "Year",
        "Lag",
        alt.Tooltip("Pounds:Q", format=",.0f"),
    ]
)

line = base.mark_line(point=True, strokeWidth=2).encode(
    y=alt.Y(
        "RONI_lagged:Q",
        title="Annual Mean RONI"
    ),
    color=alt.value("#D62728"),
    tooltip=[
        "Year",
        "Lag",
        alt.Tooltip("RONI_lagged:Q", format=".2f"),
    ]
)

chart = alt.vconcat(
    lag_text,
    alt.layer(bars, line)
    .resolve_scale(y="independent")
    .properties(
        title="California Halibut Landings and Lagged RONI",
        width=800,
        height=400
    )
)

chart

alt.VConcatChart(...)

Interactive Exploration

The previous analyses identified an approximately five-year lag as producing the strongest linear association between annual mean RONI and California halibut landings. However, exploratory visualization is most valuable when users can investigate hypotheses themselves.

The interactive visualization below allows the climate lag to be adjusted from 0 to 10 years. As the slider changes, the standardized time series and corresponding Pearson correlation update simultaneously, allowing users to explore how different lag assumptions influence the apparent relationship between climate and commercial landings.

In [187]:
# Build one standardized time-series dataset for all lag values
lagged_series = []

for lag in range(0, 11):
    roni_lag = roni_annual.copy()
    roni_lag["Year"] = roni_lag["Year"] + lag
    roni_lag = roni_lag.rename(columns={"Annual_RONI": "RONI_lagged"})

    temp = halibut.merge(roni_lag, on="Year", how="inner").copy()
    temp["Lag"] = lag

    temp["Landings_z"] = (
        temp["Pounds"] - temp["Pounds"].mean()
    ) / temp["Pounds"].std()

    temp["RONI_z"] = (
        temp["RONI_lagged"] - temp["RONI_lagged"].mean()
    ) / temp["RONI_lagged"].std()

    long_temp = temp.melt(
        id_vars=["Year", "Lag"],
        value_vars=["Landings_z", "RONI_z"],
        var_name="Variable",
        value_name="Z_score"
    )

    long_temp["Variable"] = long_temp["Variable"].replace({
        "Landings_z": "California Halibut Landings",
        "RONI_z": "Annual Mean RONI"
    })

    lagged_series.append(long_temp)

all_lagged_series = pd.concat(lagged_series, ignore_index=True)

In [188]:
lag_slider = alt.param(
    name="lag_years",
    value=7,
    bind=alt.binding_range(
        min=0,
        max=10,
        step=1,
        name="Climate lag in years: "
    )
)

lag_label = (
    alt.Chart(all_lagged_series)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .transform_aggregate(groupby=["Lag"])
    .transform_calculate(
        label="'Selected lag: ' + datum.Lag + ' years'"
    )
    .mark_text(align="left", fontSize=16, fontWeight="bold")
    .encode(text="label:N")
    .properties(width=800, height=30)
)

line_chart = (
    alt.Chart(all_lagged_series)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .mark_line(point=True)
    .encode(
        x=alt.X(
            "Year:O",
            title="Landing Year",
            axis=alt.Axis(labelAngle=-45)
        ),
        y=alt.Y(
            "Z_score:Q",
            title="Standardized Value (z-score)"
        ),
        color=alt.Color(
            "Variable:N",
            title=""
        ),
        tooltip=[
            "Year",
            "Lag",
            "Variable",
            alt.Tooltip("Z_score:Q", format=".2f")
        ]
    )
    .properties(
        title="California Halibut Landings and Lagged RONI",
        width=800,
        height=400
    )
)

chart = alt.vconcat(lag_label, line_chart)

chart

alt.VConcatChart(...)

In [189]:
# Pearson correlation for each lag
corr_df = []

for lag in range(0, 11):
    temp = all_lagged_series[
        (all_lagged_series["Lag"] == lag)
    ].pivot_table(
        index="Year",
        columns="Variable",
        values="Z_score"
    ).dropna()

    r = temp["California Halibut Landings"].corr(
        temp["Annual Mean RONI"]
    )

    corr_df.append({
        "Lag": lag,
        "Pearson_r": r
    })

corr_df = pd.DataFrame(corr_df)
corr_df

,Lag,Pearson_r
0,0,0.107383
1,1,0.069386
2,2,0.148665
3,3,0.038282
4,4,0.192426
5,5,0.299710
6,6,0.281690
7,7,0.270220
8,8,0.129822
9,9,0.133494


In [190]:
all_lagged_series_corr = all_lagged_series.merge(
    corr_df,
    on="Lag",
    how="left"
)

In [191]:
lag_label = (
    alt.Chart(all_lagged_series_corr)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .transform_aggregate(
        Pearson_r="mean(Pearson_r)",
        groupby=["Lag"]
    )
    .transform_calculate(
        label="'Selected lag: ' + datum.Lag + ' years | Pearson r = ' + format(datum.Pearson_r, '.2f')"
    )
    .mark_text(align="left", fontSize=16, fontWeight="bold")
    .encode(text="label:N")
    .properties(width=800, height=30)
)

In [192]:
line_chart = (
    alt.Chart(all_lagged_series_corr)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .mark_line(point=True)
    .encode(
        x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("Z_score:Q", title="Standardized Value (z-score)"),
        color=alt.Color("Variable:N", title=""),
        tooltip=[
            "Year",
            "Lag",
            "Variable",
            alt.Tooltip("Z_score:Q", format=".2f"),
            alt.Tooltip("Pearson_r:Q", title="Pearson r", format=".2f")
        ]
    )
    .properties(
        title="California Halibut Landings and Lagged RONI",
        width=800,
        height=400
    )
)

chart = alt.vconcat(lag_label, line_chart)
chart

alt.VConcatChart(...)

In [204]:
band_df = pd.DataFrame({
    "Year": sorted(all_lagged_series_corr["Year"].unique())
})

band_df["warm_min"] = 0
band_df["warm_max"] = 2
band_df["cool_min"] = -2
band_df["cool_max"] = 0

warm_band = alt.Chart(band_df).mark_area(
    color="#d73027",
    opacity=0.16
).encode(
    x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("warm_min:Q"),
    y2="warm_max:Q"
)

cool_band = alt.Chart(band_df).mark_area(
    color="#4575b4",
    opacity=0.16
).encode(
    x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("cool_min:Q"),
    y2="cool_max:Q"
)

line_chart = (
    alt.Chart(all_lagged_series_corr)
    .add_params(lag_slider)
    .transform_filter(alt.datum.Lag == lag_slider)
    .mark_line(point=True)
    .encode(
        x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("Z_score:Q", title="Standardized Value (z-score)"),
        color=alt.Color("Variable:N", title=""),
        tooltip=[
            "Year",
            "Lag",
            "Variable",
            alt.Tooltip("Z_score:Q", format=".2f"),
            alt.Tooltip("Pearson_r:Q", title="Pearson r", format=".2f")
        ]
    )
)

chart = alt.vconcat(
    lag_label,
    (warm_band + cool_band + line_chart).properties(
        title="California Halibut Landings and Lagged RONI",
        width=800,
        height=400
    )
)

chart

alt.VConcatChart(...)

In [209]:
warm_band = alt.Chart(band_df).mark_area(
    color="#ff6b6b",
    opacity=0.22
).encode(
    x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("warm_min:Q"),
    y2="warm_max:Q"
)

cool_band = alt.Chart(band_df).mark_area(
    color="#5dade2",
    opacity=0.22
).encode(
    x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
    y=alt.Y("cool_min:Q"),
    y2="cool_max:Q"
)

landings_line = (
    alt.Chart(all_lagged_series_corr)
    .add_params(lag_slider)
    .transform_filter(
        (alt.datum.Lag == lag_slider) &
        (alt.datum.Variable == "California Halibut Landings")
    )
    .mark_line(point=True, strokeWidth=2.5)
    .encode(
        x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("Z_score:Q", title="Standardized Value (z-score)"),
        color=alt.Color(
            "Variable:N",
            scale=alt.Scale(
                domain=[
                    "California Halibut Landings",
                    "Annual Mean RONI"
                ],
                range=[
                    "#f28e2b",
                    "#01060C"
                ]
            ),
            legend=alt.Legend(title="", orient="top-right")
        ),
        tooltip=[
            "Year", "Lag", "Variable",
            alt.Tooltip("Z_score:Q", format=".2f"),
            alt.Tooltip("Pearson_r:Q", title="Pearson r", format=".2f")
        ]
    )
)

roni_line = (
    alt.Chart(all_lagged_series_corr)
    .add_params(lag_slider)
    .transform_filter(
        (alt.datum.Lag == lag_slider) &
        (alt.datum.Variable == "Annual Mean RONI")
    )
    .mark_line(point=True, strokeWidth=2.5, strokeDash=[6, 4])
    .encode(
        x=alt.X("Year:O", title="Landing Year", axis=alt.Axis(labelAngle=-45)),
        y=alt.Y("Z_score:Q", title="Standardized Value (z-score)"),
        color=alt.Color(
            "Variable:N",
            scale=alt.Scale(
                domain=[
                    "California Halibut Landings",
                    "Annual Mean RONI"
                ],
                range=[
                    "#f28e2b",
                    "#00070E"
                ]
            ),
            legend=alt.Legend(title="", orient="top-right")
        ),
        tooltip=[
            "Year", "Lag", "Variable",
            alt.Tooltip("Z_score:Q", format=".2f"),
            alt.Tooltip("Pearson_r:Q", title="Pearson r", format=".2f")
        ]
    )
)

chart = alt.vconcat(
    lag_label,
    (warm_band + cool_band + landings_line + roni_line).properties(
        title="California Halibut Landings and Lagged RONI",
        width=800,
        height=400
    )
).configure_legend(
    orient="top",
    direction="horizontal",
    labelFontSize=12,
    titleFontSize=12
)

chart

alt.VConcatChart(...)

In [193]:
temp = all_lagged_series[
    all_lagged_series["Lag"] == 7
].pivot_table(
    index="Year",
    columns="Variable",
    values="Z_score"
).dropna()

temp.index.min(), temp.index.max(), len(temp)

(np.int64(1980), np.int64(2025), 46)

In [199]:
# Create a lookup table from the RONI rows only
climate_lookup = (
    all_lagged_series_corr[
        all_lagged_series_corr["Variable"] == "Annual Mean RONI"
    ][["Year", "Lag", "Z_score"]]
    .copy()
)

climate_lookup["Climate"] = np.where(
    climate_lookup["Z_score"] >= 0,
    "Warm",
    "Cool"
)

# Merge the climate classification back onto every row
all_lagged_series_corr = all_lagged_series_corr.merge(
    climate_lookup[["Year", "Lag", "Climate"]],
    on=["Year", "Lag"],
    how="left"
)

Conclusions

This exploratory analysis suggests that California halibut landings are more closely associated with historical ocean conditions than with simultaneous climate. Accounting for biologically realistic delays substantially strengthened the observed relationship, with the highest correlation occurring at approximately five years and elevated correlations persisting through approximately seven years. These findings are consistent with the expectation that climate influences recruitment, growth, and eventual participation in the commercial fishery over multiple years.

The interactive lag visualization provides a flexible framework for exploring delayed climate responses and serves as a prototype for extending this analysis to additional California fisheries.

In [194]:
top50 = (
    mfde.groupby("Species Name", as_index=False)["Pounds"]
    .sum()
    .sort_values("Pounds", ascending=False)
    .head(50)
)

top50

,Species Name,Pounds
324,"Squid, market",5.215393e+09
234,"Sardine, Pacific",1.963841e+09
126,"Mackerel, Pacific",1.133691e+09
359,"Tuna, yellowfin",1.092495e+09
356,"Tuna, skipjack",8.701055e+08
...,...,...
91,Grenadier,1.548744e+07
150,"Prawn, spot",1.503779e+07
298,"Smelt, night",1.338818e+07
196,"Rockfish, group small",1.332140e+07


In [195]:
top50["Species Name"].tolist()

['Squid, market',
 'Sardine, Pacific',
 'Mackerel, Pacific',
 'Tuna, yellowfin',
 'Tuna, skipjack',
 'Anchovy, northern',
 'Sea urchin, red',
 'Crab, Dungeness',
 'Sole, Dover',
 'Mackerel, unspecified',
 'Mackerel, jack',
 'Sablefish',
 'Whiting, Pacific',
 'Rockfish, unspecified',
 'Tuna, albacore',
 'Herring, Pacific',
 'Shrimp, ocean (pink)',
 'Salmon, Chinook',
 'Bonito, Pacific',
 'Thornyheads',
 'Herring, Pacific - roe',
 'Rockfish, widow',
 'Swordfish',
 'Tuna, bluefin',
 'Sole, petrale',
 'Rockfish, group bocaccio/chili',
 'Sole, English',
 'Rockfish, bocaccio',
 'Thornyhead, longspine',
 'Rockfish, chilipepper',
 'Crab, rock unspecified',
 'Lingcod',
 'Halibut, California',
 'Sole, rex',
 'Sanddab',
 'Lobster, California spiny',
 'Rockfish, group red',
 'Hagfishes',
 'Thornyhead, shortspine',
 'Shark, thresher',
 'Rockfish, group rosefish',
 'Tuna, bigeye',
 'Croaker, white',
 'Prawn, ridgeback',
 'Skate, unspecified',
 'Grenadier',
 'Prawn, spot',
 'Smelt, night',
 'Rockfish